# Praktikum Dinamika Kapal — Pertemuan 6
## Response Spectrum dari Data RAO CFD (λ/L)

Versi ini disesuaikan dengan format data CFD yang menggunakan **λ/L** sebagai sumbu-x.

Alur utama:

\[
\lambda/L \rightarrow \lambda \rightarrow k \rightarrow \omega \rightarrow \omega_e
\rightarrow S_\zeta(\omega) \rightarrow S_{\zeta e}(\omega_e)
\rightarrow S_R(\omega_e)
\]

Konversi mengikuti file MATLAB sumber:

\[
\lambda=(\lambda/L)L_{PP}
\]

\[
k=\frac{2\pi}{\lambda}
\]

\[
\omega=\sqrt{gk}
\]

\[
\omega_e=\omega\left(1-\frac{\omega V\cos\beta}{g}\right)
\]

> Jalankan cell berurutan. Notebook juga menyediakan **kalkulator λ/L ↔ λ ↔ k ↔ ω ↔ T ↔ ωe ↔ Te** sebelum analisis spektrum.


## 1. Import library

In [1]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
from google.colab import files
from IPython.display import display

g = 9.81
plt.rcParams["figure.figsize"] = (8, 4.8)
plt.rcParams["axes.grid"] = True
print("Library siap.")


Library siap.


## 2. Kalkulator konversi λ/L

Bagian ini dapat digunakan **secara mandiri**, sebelum upload RAO.

Masukkan:
- λ/L;
- panjang referensi kapal \(L\) — untuk kasus MATLAB sumber digunakan **LPP**;
- kecepatan kapal;
- heading.

Hasil kalkulator menampilkan λ, k, ω, T, ωe, dan Te.


In [2]:
# ===== KALKULATOR: EDIT INPUT =====
lambda_over_L = 1.50
L_ref = 8.23             # m; gunakan LPP
speed_calc = 10.0
speed_unit_calc = "knot" # "knot" atau "m/s"
heading_calc = 180.0     # deg
# ==================================

V_calc = speed_calc*0.514444 if speed_unit_calc == "knot" else speed_calc
beta_calc = np.deg2rad(heading_calc)

wavelength = lambda_over_L * L_ref
k = 2*np.pi / wavelength
omega = np.sqrt(g*k)
T = 2*np.pi / omega
omega_e = omega*(1 - omega*V_calc*np.cos(beta_calc)/g)
T_e = 2*np.pi/omega_e if omega_e != 0 else np.inf

calc = pd.DataFrame({
    "Parameter":["λ/L","L reference (m)","λ (m)","k (rad/m)",
                 "ω (rad/s)","T (s)","V (m/s)","Heading (deg)",
                 "ωe (rad/s)","Te (s)"],
    "Value":[lambda_over_L,L_ref,wavelength,k,omega,T,V_calc,
             heading_calc,omega_e,T_e]
})
display(calc)


,Parameter,Value
0,λ/L,1.500000
1,L reference (m),8.230000
2,λ (m),12.345000
3,k (rad/m),0.508966
4,ω (rad/s),2.234492
5,T (s),2.811907
6,V (m/s),5.144440
7,Heading (deg),180.000000
8,ωe (rad/s),4.852838
9,Te (s),1.294745


## 3. Upload file RAO CFD

Notebook mendeteksi pemisah **TAB, comma, semicolon, atau whitespace** secara otomatis.

Contoh file Anda:

| λ/L | Heave |
|---:|---:|
| 16.557051 | 1.003991 |
| 9.822667 | 1.007247 |
| 6.647498 | 1.010852 |

Jadi file tidak harus berupa CSV dengan koma.


In [ ]:
uploaded = files.upload()
filename = next(iter(uploaded))
raw = uploaded[filename]

if filename.lower().endswith((".xlsx",".xls")):
    df = pd.read_excel(io.BytesIO(raw))
else:
    # sep=None + python engine melakukan delimiter sniffing.
    df = pd.read_csv(io.BytesIO(raw), sep=None, engine="python")

# fallback untuk file whitespace/tab yang masih terbaca satu kolom
if df.shape[1] == 1 and not filename.lower().endswith((".xlsx",".xls")):
    df = pd.read_csv(io.BytesIO(raw), sep=r"\s+|\t|;|,", engine="python")

# rapikan nama kolom
df.columns = [str(c).strip() for c in df.columns]

print("File       :", filename)
print("Jumlah data:", len(df))
print("Kolom      :", list(df.columns))
display(df.head(20))


## 4. Tentukan jenis respons dan basis sumbu-x

Pilihan basis data:
- `"lambda/L"` — tipikal hasil CFD;
- `"wavelength"` — λ dalam m;
- `"wave_frequency"` — ω dalam rad/s;
- `"encounter_frequency"` — ωe dalam rad/s.

Jika menggunakan **λ/L**, notebook akan melakukan konversi yang sama seperti MATLAB sumber.


In [ ]:
# ===== EDIT BAGIAN INI =====
response_type = "Heave"      # "Heave", "Pitch", "Added Resistance"
x_column = "λ/L"             # nama kolom sumbu-x pada file
rao_column = "Heave"         # nama kolom RAO/operator
x_basis = "lambda/L"         # lambda/L, wavelength, wave_frequency, encounter_frequency
# ===========================

if x_column not in df.columns:
    raise ValueError(f"Kolom '{x_column}' tidak ditemukan. Kolom tersedia: {list(df.columns)}")
if rao_column not in df.columns:
    raise ValueError(f"Kolom '{rao_column}' tidak ditemukan. Kolom tersedia: {list(df.columns)}")

x_raw = pd.to_numeric(df[x_column], errors="coerce").to_numpy()
rao_raw = pd.to_numeric(df[rao_column], errors="coerce").to_numpy()
ok = np.isfinite(x_raw) & np.isfinite(rao_raw)
x_raw, rao_raw = x_raw[ok], rao_raw[ok]

print("Response :", response_type)
print("X basis  :", x_basis)
print("X column :", x_column)
print("RAO      :", rao_column)
print("N        :", len(x_raw))


## 5. Input kapal dan kondisi operasi

**Penting:** panjang referensi harus konsisten dengan definisi λ/L dari CFD.  
Pada MATLAB sumber Anda, perhitungannya menggunakan:

```text
wave_length = wave_ratio .* LPP
```

maka default di sini juga menggunakan **LPP**.

Konvensi heading juga dipertahankan: **180° = head sea**.


In [ ]:
# ===== EDIT BAGIAN INI =====
LPP = 8.23              # m
speed = 10.0
speed_unit = "knot"     # "knot" atau "m/s"
heading_deg = 180.0
# ===========================

V = speed*0.514444 if speed_unit == "knot" else speed
beta = np.deg2rad(heading_deg)

print(f"LPP     = {LPP:.4f} m")
print(f"V       = {V:.4f} m/s")
print(f"Heading = {heading_deg:.1f} deg")


## 6. Konversi seluruh data RAO ke frekuensi

Untuk data CFD berbasis λ/L:

\[
\lambda=(\lambda/L)L_{PP},\qquad
k=\frac{2\pi}{\lambda},\qquad
\omega=\sqrt{gk}
\]

Kemudian:

\[
\omega_e=\omega\left(1-\frac{\omega V\cos\beta}{g}\right)
\]

Tabel konversi ditampilkan agar mahasiswa dapat memeriksa setiap tahap.


In [ ]:
if x_basis == "lambda/L":
    lam_ratio = x_raw
    lam = lam_ratio * LPP
    k_data = 2*np.pi/lam
    omega_data = np.sqrt(g*k_data)
    omega_e_data = omega_data*(1 - omega_data*V*np.cos(beta)/g)

elif x_basis == "wavelength":
    lam = x_raw
    lam_ratio = lam/LPP
    k_data = 2*np.pi/lam
    omega_data = np.sqrt(g*k_data)
    omega_e_data = omega_data*(1 - omega_data*V*np.cos(beta)/g)

elif x_basis == "wave_frequency":
    omega_data = x_raw
    k_data = omega_data**2/g
    lam = 2*np.pi/k_data
    lam_ratio = lam/LPP
    omega_e_data = omega_data*(1 - omega_data*V*np.cos(beta)/g)

elif x_basis == "encounter_frequency":
    # ω tidak dapat dipulihkan secara unik untuk semua kondisi hanya dari ωe.
    omega_e_data = x_raw
    omega_data = np.full_like(x_raw, np.nan)
    k_data = np.full_like(x_raw, np.nan)
    lam = np.full_like(x_raw, np.nan)
    lam_ratio = np.full_like(x_raw, np.nan)

else:
    raise ValueError("x_basis tidak dikenali.")

T_data = np.where(np.isfinite(omega_data), 2*np.pi/omega_data, np.nan)
Te_data = np.where(omega_e_data != 0, 2*np.pi/omega_e_data, np.nan)

conversion = pd.DataFrame({
    "lambda/L":lam_ratio,
    "lambda_m":lam,
    "k_rad_m":k_data,
    "omega_rad_s":omega_data,
    "T_s":T_data,
    "omega_e_rad_s":omega_e_data,
    "Te_s":Te_data,
    "RAO_or_operator":rao_raw
})
display(conversion)


## 7. Plot RAO terhadap λ/L dan encounter frequency

Dua grafik ditampilkan agar hubungan antara **domain CFD** dan **domain analisis spektrum** terlihat jelas.


In [ ]:
if np.any(np.isfinite(lam_ratio)):
    plt.figure()
    plt.plot(lam_ratio, rao_raw, "o-")
    plt.xlabel("λ/L")
    plt.ylabel("RAO / response operator")
    plt.title(f"{response_type} RAO — CFD Domain")
    plt.show()

order = np.argsort(omega_e_data)
we_rao = omega_e_data[order]
rao_sorted = rao_raw[order]

plt.figure()
plt.plot(we_rao, rao_sorted, "o-")
plt.xlabel("Encounter frequency, ωe (rad/s)")
plt.ylabel("RAO / response operator")
plt.title(f"{response_type} RAO — Encounter Frequency Domain")
plt.show()


## 8. Pilih wave spectrum

Tersedia:
- JONSWAP — formula dipertahankan dari MATLAB sumber;
- Bretschneider;
- Pierson–Moskowitz.

Untuk JONSWAP, default \(\gamma=2.5\), sama dengan MATLAB sumber.


In [ ]:
# ===== EDIT BAGIAN INI =====
spectrum_type = "JONSWAP"   # JONSWAP, Bretschneider, Pierson-Moskowitz
Hs = 1.25                    # m
Tp = 5.5                     # s
gamma = 2.5
# ===========================

def jonswap_matlab(omega, Hs, Tp, gamma=2.5):
    wp = 2*np.pi/Tp
    sigma = np.where(omega <= wp, 0.07, 0.09)
    A = np.exp(-((omega/wp - 1)/(sigma*np.sqrt(2)))**2)
    return ((320*Hs**2)/Tp**4)*omega**(-5)*np.exp(
        -(5/4)*(omega/wp)**(-4)
    )*gamma**A

def bretschneider(omega, Hs, Tp):
    wp = 2*np.pi/Tp
    return (5/16)*Hs**2*wp**4*omega**(-5)*np.exp(
        -(5/4)*(wp/omega)**4
    )

def pierson_moskowitz(omega, Hs, Tp):
    return bretschneider(omega, Hs, Tp)

print("Spectrum =", spectrum_type)
print("Hs =", Hs, "m")
print("Tp =", Tp, "s")
if spectrum_type == "JONSWAP":
    print("gamma =", gamma)


## 9. Hitung wave spectrum \(S_\zeta(\omega)\)

Untuk data λ/L, domain wave frequency diambil dari rentang \(\omega\) hasil konversi RAO sehingga domain perhitungan konsisten.


In [ ]:
if np.all(~np.isfinite(omega_data)):
    # Bila input hanya ωe, gunakan domain gelombang umum.
    omega_grid = np.linspace(0.15, 4.0, 800)
else:
    omin = np.nanmin(omega_data)
    omax = np.nanmax(omega_data)
    omega_grid = np.linspace(omin, omax, 600)

if spectrum_type == "JONSWAP":
    S_wave = jonswap_matlab(omega_grid, Hs, Tp, gamma)
elif spectrum_type == "Bretschneider":
    S_wave = bretschneider(omega_grid, Hs, Tp)
elif spectrum_type == "Pierson-Moskowitz":
    S_wave = pierson_moskowitz(omega_grid, Hs, Tp)
else:
    raise ValueError("Spectrum tidak dikenali.")

plt.figure()
plt.plot(omega_grid, S_wave, linewidth=2)
plt.xlabel("Wave frequency, ω (rad/s)")
plt.ylabel("Sζ(ω) [m²/(rad/s)]")
plt.title(f"{spectrum_type} — Hs={Hs} m, Tp={Tp} s")
plt.show()


## 10. Encounter wave spectrum

Mengikuti MATLAB sumber:

\[
S_{\zeta e}(\omega_e)=
\frac{S_\zeta(\omega)}
{\left|1-\frac{2\omega V\cos\beta}{g}\right|}
\]


In [ ]:
omega_e_grid = omega_grid*(1 - omega_grid*V*np.cos(beta)/g)
den = 1 - 2*omega_grid*V*np.cos(beta)/g

with np.errstate(divide="ignore", invalid="ignore"):
    S_enc = S_wave/np.abs(den)

valid = np.isfinite(omega_e_grid) & np.isfinite(S_enc) & (omega_e_grid > 0) & (S_enc >= 0)

enc = pd.DataFrame({
    "omega":omega_grid[valid],
    "omega_e":omega_e_grid[valid],
    "S_wave":S_wave[valid],
    "S_enc":S_enc[valid]
}).sort_values("omega_e").groupby("omega_e", as_index=False).mean()

plt.figure()
plt.plot(enc["omega_e"], enc["S_enc"], linewidth=2)
plt.xlabel("Encounter frequency, ωe (rad/s)")
plt.ylabel("Sζe(ωe) [m²/(rad/s)]")
plt.title("Encounter Wave Spectrum")
plt.show()


## 11. Interpolasi RAO pada domain encounter spectrum

Seperti MATLAB `spline(...)`, digunakan cubic spline. Perhitungan hanya dilakukan pada **daerah overlap** antara RAO dan encounter wave spectrum.


In [ ]:
rao_table = pd.DataFrame({"we":omega_e_data, "rao":rao_raw})
rao_table = rao_table.replace([np.inf,-np.inf],np.nan).dropna()
rao_table = rao_table.groupby("we",as_index=False)["rao"].mean().sort_values("we")

we0 = rao_table["we"].to_numpy()
r0 = rao_table["rao"].to_numpy()

fmin = max(we0.min(), enc["omega_e"].min())
fmax = min(we0.max(), enc["omega_e"].max())

if fmax <= fmin:
    raise ValueError("Domain RAO dan encounter spectrum tidak overlap.")

we_common = np.linspace(fmin, fmax, 400)
rao_interp = CubicSpline(we0, r0, extrapolate=False)(we_common)
S_enc_interp = np.interp(we_common, enc["omega_e"], enc["S_enc"])

plt.figure()
plt.scatter(we0, r0, label="CFD / imported RAO")
plt.plot(we_common, rao_interp, linewidth=2, label="Cubic spline")
plt.xlabel("Encounter frequency, ωe (rad/s)")
plt.ylabel("RAO / response operator")
plt.title(f"{response_type} RAO Interpolation")
plt.legend()
plt.show()


## 12. Response spectrum

Untuk **Heave/Pitch**:

\[
S_R=RAO^2S_{\zeta e}
\]

Untuk **Added Resistance**, mengikuti MATLAB sumber:

\[
S_{AR}=RAO_{AR}S_{\zeta e}
\]


In [ ]:
if response_type in ["Heave","Pitch"]:
    S_response = rao_interp**2 * S_enc_interp
elif response_type == "Added Resistance":
    S_response = rao_interp * S_enc_interp
else:
    raise ValueError("response_type harus Heave, Pitch, atau Added Resistance.")

plt.figure()
plt.plot(we_common, S_response, linewidth=2)
plt.xlabel("Encounter frequency, ωe (rad/s)")
plt.ylabel("Response spectral density")
plt.title(f"{response_type} Response Spectrum")
plt.show()


## 13. Respons Posisi, Kecepatan, dan Percepatan

Sesuai **HResP**, untuk gerak translasi seperti **heave**, response spectrum posisi adalah:

\[
S_{\eta}(\omega_e)=|RAO(\omega_e)|^2 S_{\zeta e}(\omega_e)
\]

Spektrum respons kecepatan diperoleh dengan mengalikan \(\omega_e^2\):

\[
S_{\dot{\eta}}(\omega_e)=\omega_e^2 S_{\eta}(\omega_e)
\]

dan spektrum respons percepatan:

\[
S_{\ddot{\eta}}(\omega_e)=\omega_e^4 S_{\eta}(\omega_e)
\]

Sehingga spectral moments yang digunakan adalah:

\[
m_0=\int S_{\eta}(\omega_e)d\omega_e
\]

\[
m_2=\int \omega_e^2 S_{\eta}(\omega_e)d\omega_e
\]

\[
m_4=\int \omega_e^4 S_{\eta}(\omega_e)d\omega_e
\]

dan:

\[
RMS_{\eta}=\sqrt{m_0},\qquad
RMS_{\dot{\eta}}=\sqrt{m_2},\qquad
RMS_{\ddot{\eta}}=\sqrt{m_4}
\]

Untuk **pitch**, bentuk matematisnya sama, tetapi respons posisi adalah **angular displacement**, kecepatan adalah **angular velocity**, dan percepatan adalah **angular acceleration**. Satuan mengikuti satuan RAO pitch yang diimpor.

Untuk **Added Resistance**, turunan posisi–kecepatan–percepatan tidak digunakan. Program tetap menghitung mean added resistance mengikuti kode MATLAB sumber.


In [ ]:
# ============================================================
# POSITION / VELOCITY / ACCELERATION RESPONSE — HResP METHOD
# ============================================================

if response_type in ["Heave", "Pitch"]:

    # Response spectrum posisi / angular displacement
    S_position = S_response

    # Response spectrum kecepatan dan percepatan
    S_velocity = we_common**2 * S_position
    S_acceleration = we_common**4 * S_position

    # Spectral moments
    m0 = np.trapezoid(S_position, we_common)
    m2 = np.trapezoid(S_velocity, we_common)
    m4 = np.trapezoid(S_acceleration, we_common)

    # RMS
    RMS_position = np.sqrt(max(m0, 0))
    RMS_velocity = np.sqrt(max(m2, 0))
    RMS_acceleration = np.sqrt(max(m4, 0))

    print("="*65)
    print("HASIL RESPONS POSISI, KECEPATAN, DAN PERCEPATAN")
    print("="*65)
    print(f"Response : {response_type}")
    print(f"LPP      : {LPP:.4f} m")
    print(f"Speed    : {V:.4f} m/s")
    print(f"Heading  : {heading_deg:.1f} deg")
    print(f"Spectrum : {spectrum_type}")
    print(f"Hs / Tp  : {Hs:.3f} m / {Tp:.3f} s")
    print("-"*65)
    print(f"m0       : {m0:.8g}")
    print(f"m2       : {m2:.8g}")
    print(f"m4       : {m4:.8g}")
    print("-"*65)

    if response_type == "Heave":
        print(f"RMS Heave Position     : {RMS_position:.8g} m")
        print(f"RMS Heave Velocity     : {RMS_velocity:.8g} m/s")
        print(f"RMS Heave Acceleration : {RMS_acceleration:.8g} m/s²")
    else:
        print(f"RMS Pitch Angle        : {RMS_position:.8g} [sesuai satuan RAO]")
        print(f"RMS Angular Velocity   : {RMS_velocity:.8g} [satuan sudut/s]")
        print(f"RMS Angular Accel.     : {RMS_acceleration:.8g} [satuan sudut/s²]")

    # Plot tiga spektrum respons secara terpisah
    plt.figure()
    plt.plot(we_common, S_position, linewidth=2)
    plt.xlabel("Encounter frequency, ωe (rad/s)")
    plt.ylabel("Position response spectral density")
    plt.title(f"{response_type} — Position / Displacement Spectrum")
    plt.show()

    plt.figure()
    plt.plot(we_common, S_velocity, linewidth=2)
    plt.xlabel("Encounter frequency, ωe (rad/s)")
    plt.ylabel("Velocity response spectral density")
    plt.title(f"{response_type} — Velocity Spectrum")
    plt.show()

    plt.figure()
    plt.plot(we_common, S_acceleration, linewidth=2)
    plt.xlabel("Encounter frequency, ωe (rad/s)")
    plt.ylabel("Acceleration response spectral density")
    plt.title(f"{response_type} — Acceleration Spectrum")
    plt.show()

else:
    m0 = np.trapezoid(S_response, we_common)
    mean_ar = 2*m0

    print("="*65)
    print("HASIL ADDED RESISTANCE")
    print("="*65)
    print(f"m0                    : {m0:.8g}")
    print(f"Mean Added Resistance : {mean_ar:.8g}")


## 14. Grafik ringkas hasil

Untuk Heave/Pitch ditampilkan **RAO, encounter wave spectrum, position spectrum, velocity spectrum, dan acceleration spectrum**.


In [ ]:
if response_type in ["Heave", "Pitch"]:
    fig, ax = plt.subplots(5,1,figsize=(9,18))

    ax[0].scatter(we0,r0,label="CFD")
    ax[0].plot(we_common,rao_interp,label="Spline")
    ax[0].set_ylabel("RAO")
    ax[0].set_title("RAO")
    ax[0].legend()

    ax[1].plot(we_common,S_enc_interp)
    ax[1].set_ylabel("Sζe")
    ax[1].set_title("Encounter Wave Spectrum")

    ax[2].plot(we_common,S_position)
    ax[2].set_ylabel("S position")
    ax[2].set_title("Position / Displacement Response Spectrum")

    ax[3].plot(we_common,S_velocity)
    ax[3].set_ylabel("S velocity")
    ax[3].set_title("Velocity Response Spectrum")

    ax[4].plot(we_common,S_acceleration)
    ax[4].set_xlabel("Encounter frequency, ωe (rad/s)")
    ax[4].set_ylabel("S acceleration")
    ax[4].set_title("Acceleration Response Spectrum")

    plt.tight_layout()
    plt.show()
else:
    fig, ax = plt.subplots(3,1,figsize=(9,12))
    ax[0].scatter(we0,r0,label="CFD")
    ax[0].plot(we_common,rao_interp,label="Spline")
    ax[0].set_title("Added Resistance Operator")
    ax[0].legend()
    ax[1].plot(we_common,S_enc_interp)
    ax[1].set_title("Encounter Wave Spectrum")
    ax[2].plot(we_common,S_response)
    ax[2].set_title("Added Resistance Spectral Contribution")
    ax[2].set_xlabel("Encounter frequency, ωe (rad/s)")
    plt.tight_layout()
    plt.show()


## 15. Export tabel konversi dan hasil

In [ ]:
if response_type in ["Heave", "Pitch"]:
    result = pd.DataFrame({
        "Encounter_Frequency_rad_s": we_common,
        "RAO_or_Operator": rao_interp,
        "Encounter_Wave_Spectrum": S_enc_interp,
        "Position_Response_Spectrum": S_position,
        "Velocity_Response_Spectrum": S_velocity,
        "Acceleration_Response_Spectrum": S_acceleration
    })
else:
    result = pd.DataFrame({
        "Encounter_Frequency_rad_s": we_common,
        "RAO_or_Operator": rao_interp,
        "Encounter_Wave_Spectrum": S_enc_interp,
        "Added_Resistance_Spectral_Contribution": S_response
    })

conversion.to_csv("RAO_frequency_conversion.csv", index=False)
result.to_csv("response_spectrum_result.csv", index=False)

print("Tersimpan:")
print("- RAO_frequency_conversion.csv")
print("- response_spectrum_result.csv")
display(result.head(20))

# Bila ingin download:
# files.download("RAO_frequency_conversion.csv")
# files.download("response_spectrum_result.csv")


## 16. Sanity Check

Sebelum menerima hasil:

1. Pastikan definisi **λ/L** sama dengan panjang referensi yang dimasukkan.
2. Untuk workflow MATLAB sumber, gunakan **LPP**.
3. Pastikan speed dan heading konsisten dengan kondisi analisis.
4. Periksa tabel konversi λ/L → λ → ω → ωe.
5. Periksa apakah peak wave spectrum sesuai \(T_p\).
6. Pastikan domain RAO dan encounter spectrum overlap.
7. Interpretasikan mengapa peak response spectrum terjadi pada frekuensi tertentu.

**Hasil software/kode tidak otomatis benar tanpa pemeriksaan kewajaran fisik dan satuan.**
